In [3]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback

# Dataset, Data Moule and Model Classes

In [4]:
class MyDataset(Dataset):
 def __init__(self, data, outputs):
  """Inicjalizacja - przygotowanie struktury danych"""
  self.data = data
  self.outputs = outputs

 def __len__(self):
  """Zwraca liczbę próbek w datasecie"""
  return len(self.data)

 def __getitem__(self, idx):
  """Zwraca próbkę o indeksie idx"""
  x = self.data[idx]
  y = self.outputs[idx]
  return x, y

class MyDataModule(pl.LightningDataModule):
 def __init__(self, batch_size=32,n_samples=5000,n_features=20,random_state=42):
  super().__init__()
  self.X, self.y = make_regression(n_samples=n_samples, n_features=n_features,
random_state=random_state)
  self.X = torch.tensor(self.X, dtype=torch.float32)
  self.y = torch.tensor(self.y, dtype=torch.float32)
  self.batch_size = batch_size

 def setup(self, stage=None):
  X_train_val, X_test, y_train_val, y_test = train_test_split(self.X, self.y, test_size=0.2)
  X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)
  self.train_dataset = MyDataset(X_train, y_train)
  self.val_dataset = MyDataset(X_val, y_val)
  self.test_dataset = MyDataset(X_test, y_test)

 def train_dataloader(self):
  return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

 def val_dataloader(self):
  return DataLoader(self.val_dataset, batch_size=self.batch_size)

 def test_dataloader(self):
  return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [5]:
class LitModel(pl.LightningModule):
 def __init__(self, input_size, hidden_size, output_dim,lr=0.001):
  super().__init__()
  self.save_hyperparameters() # Call this first to save __init__ arguments

  # to self.hparams
  self.layer1 = nn.Linear(self.hparams.input_size, self.hparams.hidden_size)
  self.layer2 = nn.Linear(self.hparams.hidden_size, self.hparams.hidden_size)
  self.layer3 = nn.Linear(self.hparams.hidden_size, self.hparams.output_dim)
  self.criterion = nn.MSELoss()

 def forward(self, x):
  x = torch.relu(self.layer1(x))
  x = torch.relu(self.layer2(x))
  x = self.layer3(x)
  return x

 def training_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  self.log('train_loss', loss)
  return loss

 def validation_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('val_loss', loss, prog_bar=True)
  self.log('val_r2', r2, prog_bar=True)

 def test_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('test_loss', loss)
  self.log('test_r2', r2)

def configure_optimizers(self):
 # Use the learning rate from self.hparams
 return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)